### 1. Install Required Libraries 

### 2. Load and Preprocess the Dataset 

In [18]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [19]:
df = pd.read_csv('AA.csv')
print(df.columns)
display(df.isna().sum().sort_values(ascending=False))
df['Target'] = df['Close'].shift(-1)
df = df.dropna(subset=['Target'])

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'], dtype='str')


Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

In [20]:
scaler = MinMaxScaler()
cols_to_scale = df.select_dtypes(include='number').columns
scaled_data = scaler.fit_transform(df[cols_to_scale])
scaled_df = pd.DataFrame(scaled_data, columns=cols_to_scale, index=df.index)
    
df

,Date,Open,High,Low,Close,Adj Close,Volume,Target
0,1962-01-02,6.532155,6.556185,6.532155,6.532155,1.536658,55900,6.632280
1,1962-01-03,6.532155,6.632280,6.524145,6.632280,1.560212,74500,6.632280
2,1962-01-04,6.632280,6.664320,6.632280,6.632280,1.560212,80500,6.624270
3,1962-01-05,6.632280,6.656310,6.616260,6.624270,1.558326,70500,6.408000
4,1962-01-08,6.608250,6.608250,6.339915,6.408000,1.507450,93800,6.355935
...,...,...,...,...,...,...,...,...
14657,2020-03-25,7.930000,7.990000,7.010000,7.090000,7.090000,12619300,6.840000
14658,2020-03-26,7.270000,7.386000,6.730000,6.840000,6.840000,11916400,6.550000
14659,2020-03-27,6.510000,6.790000,6.050000,6.550000,6.550000,10075400,6.070000
14660,2020-03-30,6.550000,6.600000,6.000000,6.070000,6.070000,9615600,6.160000


### 3. Prepare the Dataset for Training 

In [21]:
X = scaled_df.drop(columns=['Target'])
y = scaled_df['Target']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader

class StockDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.targets = torch.tensor(targets.values, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

In [23]:
train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 4. Define the LSTM Model 

In [24]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = LSTMModel(input_size=X_train.shape[1])

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, output_size=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, h_n = self.gru(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

model = GRUModel(input_size=X_train.shape[1])

### 5. Train the Model

In [25]:
import torch.optim as optim

loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [29]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for features_batch, target_batch in loader:
        features_batch = features_batch.unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(features_batch).squeeze()
        loss_value = loss_fn(outputs, target_batch)
        loss_value.backward()
        optimizer.step()

        total_loss += loss_value.item() * features_batch.size(0)

    return total_loss / len(loader.dataset)

def validate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for features_batch, target_batch in loader:
            features_batch = features_batch.unsqueeze(1)

            outputs = model(features_batch).squeeze()
            loss_value = loss_fn(outputs, target_batch)
            total_loss += loss_value.item() * features_batch.size(0)

    return total_loss / len(loader.dataset)

In [31]:
num_epochs = 20

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, loss, optimizer)
    val_loss = validate(model, val_loader, loss)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")

Epoch [1/20] - Train Loss: 0.000334 - Val Loss: 0.000476
Epoch [2/20] - Train Loss: 0.000354 - Val Loss: 0.000317
Epoch [3/20] - Train Loss: 0.000340 - Val Loss: 0.000705
Epoch [4/20] - Train Loss: 0.000350 - Val Loss: 0.000550
Epoch [5/20] - Train Loss: 0.000360 - Val Loss: 0.000790
Epoch [6/20] - Train Loss: 0.000373 - Val Loss: 0.000529
Epoch [7/20] - Train Loss: 0.000363 - Val Loss: 0.000416
Epoch [8/20] - Train Loss: 0.000351 - Val Loss: 0.000491
Epoch [9/20] - Train Loss: 0.000327 - Val Loss: 0.000586
Epoch [10/20] - Train Loss: 0.000338 - Val Loss: 0.000319
Epoch [11/20] - Train Loss: 0.000345 - Val Loss: 0.000272
Epoch [12/20] - Train Loss: 0.000360 - Val Loss: 0.000391
Epoch [13/20] - Train Loss: 0.000364 - Val Loss: 0.000554
Epoch [14/20] - Train Loss: 0.000371 - Val Loss: 0.000329
Epoch [15/20] - Train Loss: 0.000354 - Val Loss: 0.000696
Epoch [16/20] - Train Loss: 0.000357 - Val Loss: 0.000327
Epoch [17/20] - Train Loss: 0.000351 - Val Loss: 0.000477
Epoch [18/20] - Train L

### 6. Evaluate the Model 

In [32]:
from sklearn.metrics import r2_score

model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for features_batch, target_batch in test_loader:
        features_batch = features_batch.unsqueeze(1)
        outputs = model(features_batch).squeeze()

        predictions.extend(outputs.tolist())
        actuals.extend(target_batch.tolist())

r2 = r2_score(actuals, predictions)
print(f"R² score sur le test set : {r2:.4f}")

R² score sur le test set : 0.9368
